# Time series benchmarking tutorial (Hydrology CAMELS US with Chronos 2)

This notebook is the **single end-to-end file** for DS5110 final benchmarking.

- Dataset: CAMELS-US style input CSV
- Model: `amazon/chronos-2`
- Task: single-step streamflow forecasting
- Input mode: strictly univariate (`QObs(mm/d)` only)
- Lookback window: 30 days
- Split: 80% train / 10% val / 10% test per basin (chronological)
- Metric: RMSE on test set (overall)


In [ ]:
# Install dependencies (Colab-friendly)
import subprocess
import sys
from pathlib import Path

req_candidates = [
    Path('requirements.txt'),
    Path('/content/timeseries_benchmark_tutorial/requirements.txt'),
    Path('../requirements.txt'),
]

req_path = next((p for p in req_candidates if p.exists()), None)
if req_path is not None:
    print(f'Installing from: {req_path}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req_path)])
else:
    print('requirements.txt not found. Installing fallback packages...')
    subprocess.check_call([
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'pandas>=2.0',
        'numpy>=1.24',
        'matplotlib>=3.7',
        'pyarrow>=14.0',
        'chronos-forecasting>=2.0',
        'torch>=2.1',
        'tqdm>=4.66',
        'pyyaml>=6.0',
    ])

print('Dependency installation complete.')

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

# Optional Colab Drive mount:
# from google.colab import drive
# drive.mount('/content/drive')

# ----------------------------
# User-editable runtime config
# ----------------------------
INPUT_CSV_PATH = '/Users/eriche/Downloads/BasicInputTimeSeries_us.csv'  # update in Colab, e.g. '/content/BasicInputTimeSeries_us.csv'
MODEL_ID = 'amazon/chronos-2'
LOOKBACK_WINDOW = 30
FORECAST_HORIZON = 1
TRAIN_RATIO = 0.8
VAL_RATIO = 0.1
TEST_RATIO = 0.1

TIMESTAMP_COL = 'Year_Mnth_Day'
ID_COL = 'basin_id'
TARGET_COL = 'QObs(mm/d)'

OUTPUT_ROOT = Path('.')
OUTPUT_DIRS = [
    OUTPUT_ROOT / 'results',
    OUTPUT_ROOT / 'results' / 'figures',
    OUTPUT_ROOT / 'metadata',
    OUTPUT_ROOT / 'data' / 'processed',
]
for d in OUTPUT_DIRS:
    d.mkdir(parents=True, exist_ok=True)

print('Configured input path:', INPUT_CSV_PATH)
print('Output root:', OUTPUT_ROOT.resolve())

In [ ]:
import sys

# Make src/ imports work in both local and Colab contexts.
src_candidates = [Path('src'), Path('/content/timeseries_benchmark_tutorial/src'), Path('../src')]
for src in src_candidates:
    if src.exists():
        sys.path.insert(0, str(src.resolve()))
        print('Using src path:', src.resolve())
        break

from chronos import Chronos2Pipeline

from data_utils import (
    DataSpec,
    build_context_and_test,
    chronological_split_by_id,
    load_and_prepare_csv,
    to_chronos_df,
)
from chronos_eval import rolling_one_step_predictions
from metrics import rmse
from plotting import plot_actual_vs_predicted

In [ ]:
# Load and preprocess input CSV
spec = DataSpec(
    timestamp_col=TIMESTAMP_COL,
    id_col=ID_COL,
    target_col=TARGET_COL,
    drop_unnamed_first_col=True,
)

raw_df = load_and_prepare_csv(INPUT_CSV_PATH, spec)
print('Prepared rows:', len(raw_df))
print('Prepared basins:', raw_df[ID_COL].nunique())
print(raw_df.head())

In [ ]:
# Chronological split per basin: 80/10/10
splits = chronological_split_by_id(
    raw_df,
    id_col=ID_COL,
    timestamp_col=TIMESTAMP_COL,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    test_ratio=TEST_RATIO,
)

train_df = to_chronos_df(splits['train'], id_col=ID_COL, timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)
val_df = to_chronos_df(splits['val'], id_col=ID_COL, timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)
test_df = to_chronos_df(splits['test'], id_col=ID_COL, timestamp_col=TIMESTAMP_COL, target_col=TARGET_COL)

context_df, eval_test_df = build_context_and_test(train_df, val_df, test_df)

print('Train rows:', len(train_df))
print('Val rows:', len(val_df))
print('Test rows:', len(eval_test_df))
print('Context rows:', len(context_df))

In [ ]:
# Initialize Chronos-2 pipeline (GPU preferred)
import torch

device_map = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device_map)

pipeline = Chronos2Pipeline.from_pretrained(
    MODEL_ID,
    device_map=device_map,
)

In [ ]:
# Run rolling one-step predictions over test set
pred_test_df = rolling_one_step_predictions(
    pipeline=pipeline,
    context_df=context_df,
    test_df=eval_test_df,
    lookback_window=LOOKBACK_WINDOW,
    quantile_levels=[0.5],
)

if pred_test_df.empty:
    raise RuntimeError('Prediction output is empty. Check input path and splits.')

pred_test_df.head()

In [ ]:
# Compute overall test RMSE and save predictions
overall_test_rmse = rmse(pred_test_df['actual'].values, pred_test_df['predicted'].values)
print('Overall Test RMSE:', overall_test_rmse)

pred_path = OUTPUT_ROOT / 'results' / 'predictions_test.csv'
pred_test_df.to_csv(pred_path, index=False)
print('Saved:', pred_path)

In [ ]:
# Plot Actual vs Predicted
fig_path = OUTPUT_ROOT / 'results' / 'figures' / 'actual_vs_predicted.png'
plot_actual_vs_predicted(pred_test_df, str(fig_path))
print('Saved:', fig_path)

# Display in notebook
import matplotlib.pyplot as plt
img = plt.imread(fig_path)
plt.figure(figsize=(7, 7))
plt.imshow(img)
plt.axis('off')
plt.show()

In [ ]:
# Write benchmark result JSON
benchmark_result = {
    'dataset_name': 'CAMELS-US',
    'model_name': 'chronos-2',
    'target_variable': TARGET_COL,
    'lookback_window': LOOKBACK_WINDOW,
    'forecast_horizon': FORECAST_HORIZON,
    'model_parameters': {
        'model_id': MODEL_ID,
        'prediction_length': 1,
        'quantile_levels': [0.5],
        'device_map': device_map,
    },
    'rmse': float(overall_test_rmse),
    'notes': 'Overall RMSE on chronological 10% test split across all basins.',
}

benchmark_path = OUTPUT_ROOT / 'results' / 'benchmark_results.json'
with open(benchmark_path, 'w', encoding='utf-8') as f:
    json.dump(benchmark_result, f, indent=2)

print('Saved:', benchmark_path)
print(json.dumps(benchmark_result, indent=2))

In [ ]:
# Update metadata files with observed dataset counts (optional quality-of-life step)
dataset_card_path = OUTPUT_ROOT / 'metadata' / 'dataset_card.json'
ts_chars_path = OUTPUT_ROOT / 'metadata' / 'ts_characteristics.json'
experiment_cfg_path = OUTPUT_ROOT / 'metadata' / 'experiment_config.json'

if dataset_card_path.exists():
    with open(dataset_card_path, 'r', encoding='utf-8') as f:
        dataset_card = json.load(f)
    dataset_card['number_of_locations'] = int(raw_df[ID_COL].nunique())
    dataset_card['number_of_time_points'] = int(len(raw_df))
    with open(dataset_card_path, 'w', encoding='utf-8') as f:
        json.dump(dataset_card, f, indent=2)

if ts_chars_path.exists():
    with open(ts_chars_path, 'r', encoding='utf-8') as f:
        ts_chars = json.load(f)
    ts_chars['number_of_instances'] = int(raw_df[ID_COL].nunique())
    with open(ts_chars_path, 'w', encoding='utf-8') as f:
        json.dump(ts_chars, f, indent=2)

if experiment_cfg_path.exists():
    with open(experiment_cfg_path, 'r', encoding='utf-8') as f:
        exp_cfg = json.load(f)
    exp_cfg['lookback_window'] = int(LOOKBACK_WINDOW)
    exp_cfg['forecast_horizon'] = int(FORECAST_HORIZON)
    exp_cfg['evaluation_metrics'] = ['rmse']
    with open(experiment_cfg_path, 'w', encoding='utf-8') as f:
        json.dump(exp_cfg, f, indent=2)

print('Metadata refresh complete.')

## Expected outputs after running all cells

- `results/benchmark_results.json`
- `results/predictions_test.csv`
- `results/figures/actual_vs_predicted.png`
- refreshed metadata files in `metadata/`

If the default `INPUT_CSV_PATH` does not exist in Colab, upload your file and update that variable before running preprocessing cells.